In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

%matplotlib inline

In [2]:
sales = pd.read_csv("C:/Users/George/Desktop/data/Video Game Sales 2024.csv")
metacritic_games = pd.read_csv("C:/Users/George/Desktop/data/Metacritic's Best Games 2025.csv")
steam = pd.read_csv("C:/Users/George/Desktop/data/Steam Games Dataset March 2025(Cleaned).csv")

---
## 1. Sales Dataset

### 1.1 Dropping Irrelevant Columns and Duplicates

In [4]:
sales =sales.drop(columns=['img', 'last_update'])
sales = sales.drop_duplicates()
print(f"Shape after cleaning: {sales.shape}")

Shape after cleaning: (63995, 12)


### 1.2 Filtering to Games with Sales Data

In [7]:
sales_clean = sales.dropna(subset=['total_sales']).copy()
print(f"Rows with sales data: {len(sales_clean):,} out of {len(sales):,}")

Rows with sales data: 18,919 out of 63,995


### 1.3 Handling Missing Values

In [9]:
sales_clean['release_year'] = pd.to_datetime(sales_clean['release_date'], errors='coerce').dt.year
sales_clean = sales_clean.drop(columns=['release_date'])


In [10]:
sales_clean['genre'] = sales_clean['genre'].fillna('Unknown')
sales_clean['console'] = sales_clean['console'].fillna('Unknown')

In [11]:
sales_clean['release_year'] = sales_clean['release_year'].fillna(sales_clean['release_year'].median())
sales_clean['critic_score'] = sales_clean['critic_score'].fillna(sales_clean['critic_score'].median())

In [12]:
for col in ['na_sales', 'jp_sales', 'pal_sales', 'other_sales']:
    if col in sales_clean.columns:
        sales_clean[col] = sales_clean[col].fillna(0)

In [13]:
print("Missing values remaining:", sales_clean.isnull().sum().sum())

Missing values remaining: 4


### 1.4 Standardising Text Columns

In [15]:
for col in ['genre', 'console', 'publisher', 'developer']:
    if col in sales_clean.columns:
        sales_clean[col] = sales_clean[col].str.strip().str.title()

sales_clean['title'] = sales_clean['title'].str.strip().str.lower()
print("Text columns standdardised.")

Text columns standdardised.


### 1.5 Encoding Categorical Variables

In [17]:
le = LabelEncoder()
sales_clean['genre_encoded'] = le.fit_transform(sales_clean['genre'])
sales_clean['console_encoded'] = le.fit_transform(sales_clean['console'])

sales_clean['publisher'] = sales_clean['publisher'].fillna('Unknown')
sales_clean['publisher_encoded'] = le.fit_transform(sales_clean['publisher'])

print(f"Unique genres: {sales_clean['genre'].nunique()}")
print(f"Unique consoles: {sales_clean['console'].nunique()}")
print(f"Unique publishers: {sales_clean['publisher'].nunique()}")

Unique genres: 20
Unique consoles: 39
Unique publishers: 739


## 2. Metacritic Dataset

In [19]:
metacritic_games = metacritic_games.drop_duplicates()
metacritic_games = metacritic_games.dropna(subset=['metascore'])

metacritic_games['userscore'] = pd.to_numeric(metacritic_games['userscore'], errors='coerce')
metacritic_games['userscore'] = metacritic_games['userscore'].fillna(metacritic_games['userscore'].median())

metacritic_games['title'] = metacritic_games['title'].str.strip().str.lower()

print(f"Metacritic clean shape: {metacritic_games.shape}")

Metacritic clean shape: (13436, 16)


## 3. Merging Sales and Metacritic

In [21]:
merged = sales_clean.merge(
    metacritic_games[['title', 'metascore', 'userscore']],
    on='title', how='left')

matched = merged['metascore'].notna().sum()


In [22]:
print(f"Merged shape: {merged.shape}")
print(f"Matched with Metacritic: {matched:,} ({matched/len(merged)*100:.1f}%)")

Merged shape: (18919, 17)
Matched with Metacritic: 8,624 (45.6%)


## 4. Combined Success Label

The 3 compontents are first scaled to the same 0-1 range using min-max normalisation, then combined with the following weights:
- **60%** total sales - commercial performance is the primary indicator
- **25%** userscore - player sentiment is more directly tied to word of mouth sales
- **15%** metascore - professional critic reception

Games at or above the median combined score are labelled as successful. Only games that matched with Metacritic are included since review scores are required.

In [36]:
combined_df = merged.dropna(subset=['total_sales', 'metascore', 'userscore']).copy()

for col in ['total_sales', 'metascore', 'userscore']:
    col_min = combined_df[col].min()
    col_max = combined_df[col].max()
    combined_df[f'{col}_scaled'] = (combined_df[col] - col_min) / (col_max - col_min)

combined_df['combined_score'] = (
    0.60 * combined_df['total_sales_scaled'] +
    0.15 * combined_df['metascore_scaled'] +
    0.25 * combined_df['userscore_scaled'] )

combined_df['success_combined'] = (combined_df['combined_score'] >= combined_df['combined_score'].median()).astype(int)

print(f"Games with combined label: {len(combined_df):,}")
print(f"Successful:   {combined_df['success_combined'].sum():,} ({combined_df['success_combined'].mean()*100:.1f}%)")
print(f"Unsuccessful: {(combined_df['success_combined']==0).sum():,} ({(1-combined_df['success_combined'].mean())*100:.1f}%)")

Games with combined label: 8,624
Successful:   4,312 (50.0%)
Unsuccessful: 4,312 (50.0%)


### Observations

The combined score produces a balanced label across the matched subset of games. Because it factors in both sales and review scores, a game that sold well but was poorly recieved or was critically acclaimed but commercially underperformed will not automatically be labelled as successful.

---
## 5. Steam Dataset

In [41]:
steam = steam.drop_duplicates()
steam['name'] = steam['name'].str.strip().str.lower()

steam_cols = ['name', 'release', 'price', 'positive', 'negative', 'average_playtime_forever', 'peak_ccu']
steam_cols = [c for c in steam_cols if c in steam.columns]
steam_clean = steam[steam_cols].copy()

if 'positive' in steam_clean.columns and 'negative' in steam_clean.columns:
    total = steam_clean['positive'] + steam_clean['negative']
    steam_clean['review_ratio'] = (steam_clean['positive'] / total.replace(0, np.nan)).fillna(0)

print(f"Steam clean shape: {steam_clean.shape}")

Steam clean shape: (89618, 7)


---
## 6. Saving Cleaned Datasets

In [44]:
combined_df.to_csv("C:/Users/George/Desktop/data/combined_label.csv", index=False)
sales_clean.to_csv("C:/Users/George/Desktop/data/sales_clean.csv", index=False)
metacritic_games.to_csv("C:/Users/George/Desktop/data/metacritic_clean.csv", index=False)
steam_clean.to_csv("C:/Users/George/Desktop/data/steam_clean.csv", index=False)
merged.to_csv("C:/Users/George/Desktop/data/merged_sales_metacritic.csv", index=False)

print("All cleaned datases saved.")
print(f"sales_cleaned: {sales_clean.shape}")
print(f"metacritic_clean: {metacritic_games.shape}")
print(f"steam_clean: {steam_clean.shape}")
print(f"merged_sales_metacritic: {merged.shape}")

print(f"combined_label: {combined_df.shape}")

All cleaned datases saved.
sales_cleaned: (18919, 15)
metacritic_clean: (13436, 16)
steam_clean: (89618, 7)
merged_sales_metacritic: (18919, 17)
combined_label: (8624, 22)


In [46]:
combined_df['release_year'].describe()

count    8624.000000
mean     2008.314935
std         5.195367
min      1982.000000
25%      2004.000000
50%      2008.000000
75%      2012.000000
max      2020.000000
Name: release_year, dtype: float64